# Неделя 4 — Итоговый отчёт

Карточка недели: `docs/week4_experiment.md` · Методичка, раздел 8

Отчёт должен читаться человеком, который не видел ваш код.
Под каждым разделом — 1–3 абзаца и, где уместно, таблица или график.

In [ ]:
import sys

import pandas as pd

sys.path.append('..')

## 1. Постановка задачи и определение оттока

**Бизнес-задача:** Предсказать отток B2B-клиентов телеком-оператора за 3 месяца до фактического ухода, чтобы отдел удержания мог проактивно связаться с клиентами из группы риска и предложить им индивидуальные условия.

**Определение оттока:** Клиент считается ушедшим (целевая переменная `target = 1`), если выполняется хотя бы одно из двух условий:
1. **Явный отток:** В таблице клиентов заполнено поле `termination_date`.
2. **Тихий отток:** Поле `termination_date` пустое, но средняя выручка (`revenue`) клиента за последние 3 месяца составляет менее 20% от его исторической средней выручки. Это позволяет отловить клиентов, которые фактически перестали пользоваться услугами, но формально не расторгли договор.

**Целевая метрика:** Мы фокусируемся на ранжирующих метриках (ROC-AUC, Lift@10%), так как задача состоит не в том, чтобы угадать конкретный порог, а в том, чтобы максимально точно выстроить клиентов в ряд от наиболее рискованных к наименее рискованным.

## 2. Данные: объём, период, доля оттока по сегментам

**Объём данных:**
- Справочник клиентов (`clients`): 50 400 уникальных клиентов.
- История потребления (`usage`): 1 073 079 записей о помесячном потреблении.

**Период наблюдения:** Данные охватывают 30 месяцев (с сентября 2023 года по июнь 2026 года).

**Доля оттока:**
Общая доля ушедших клиентов составляет **~29.8%** (включая ~1 000 клиентов с "тихим оттоком"). 

Распределение оттока по сегментам бизнеса неравномерно:
| Сегмент | Количество клиентов | Доля оттока |
|---------|---------------------|-------------|
| Малый бизнес | 24 388 | **30.88%** |
| Средний бизнес | 17 079 | **27.60%** |
| Крупный бизнес | 8 933 | **20.11%** |

**Вывод:** Малый бизнес уходит в 1.5 раза чаще крупного, что логично: у крупных компаний выше издержки переключения и долгосрочные контракты.

## 3. Схема сборки выборки: снапшоты, окна, зазор

Для предотвращения утечки данных (data leakage) использовано строгое временное разбиение **out-of-time** с обязательным зазором (embargo):

| Снапшот | Окно наблюдения (признаки) | Окно прогноза (таргет) | Роль в обучении |
|---------|----------------------------|------------------------|-----------------|
| 2024-07 | 2024-02 … 2024-07 | 2024-10 … 2024-12 | Train |
| 2024-10 | 2024-05 … 2024-10 | 2025-01 … 2025-03 | Train |
| 2025-01 | 2024-08 … 2025-01 | 2025-04 … 2025-06 | **Valid** (для early stopping) |
| 2025-04 | — | — | *Зазор (не используется)* |
| 2025-07 | — | — | *Зазор (не используется)* |
| 2025-10 | — | — | *Зазор (не используется)* |
| 2026-01 | 2025-08 … 2026-01 | 2026-04 … 2026-06 | **Test** (финальная оценка) |

**Зачем нужен зазор?** Три снапшота (2025-04, 2025-07, 2025-10) намеренно выброшены. Это гарантирует, что окно наблюдения тестовой выборки (начинается в 2025-08) ни на один месяц не пересекается с последним окном прогноза валидации (заканчивается в 2025-06). Тестовая выборка используется **ровно один раз** для финальной оценки победителя.

## 4. Признаки: группы и интуиция

Модель использует около 20 числовых и 3 категориальных признака, сгруппированных по бизнес-логике:

1. **Платежи:** `revenue_mean`, `revenue_last`, `revenue_last_to_mean`. *Интуиция:* Резкое падение выручки относительно среднего уровня — самый ранний и сильный сигнал оттока.
2. **Динамика:** `revenue_trend` (наклон тренда), `traffic_last_to_mean`. *Интуиция:* Нестабильность потребления и снижение трафика часто предшествуют расторжению договора.
3. **Объём отношений:** `n_sim_last`, `n_sim_trend`. *Интуиция:* Сокращение числа активных SIM-карт указывает на сворачивание активности клиента с оператором.
4. **Сигналы боли:** `n_tickets_sum`, `debt_months`. *Интуиция:* Рост обращений в поддержку и накопление долга являются прямыми индикаторами неудовлетворённости сервисом.
5. **Категориальные:** `segment`, `product`, `region`. *Интуиция:* Базовая принадлежность клиента влияет на априорную вероятность оттока.

## 5. Схема валидации и метрики

**Схема валидации:** Out-of-time split с зазором (см. раздел 3). Все сравнения моделей и подбор гиперпараметров (early stopping) проводятся строго на валидационном снапшоте (2025-01).

**Выбранные метрики:**
- **ROC-AUC:** Оценивает способность модели ранжировать клиентов по риску (0.5 — случайный выбор, 1.0 — идеальное ранжирование). Не чувствительна к дисбалансу классов.
- **PR-AUC:** Более чувствительна к качеству предсказания миноритарного класса (оттока).
- **Precision@10%:** Доля реальных уходов среди топ-10% клиентов, отсортированных по предсказанному риску.
- **Lift@10%:** Показывает, во сколько раз чаще мы находим ушедших в топ-10% по сравнению со случайным выбором.

*Почему не Accuracy?* При доле оттока ~30% модель, которая всегда предсказывает "не уйдёт", даст accuracy 70%, но будет абсолютно бесполезна для бизнеса. Ранжирующие метрики корректно оценивают качество в условиях дисбаланса.

## 6. Результаты моделей: правило → логистическая регрессия → CatBoost

Сравнение трёх подходов возрастающей сложности на тестовой выборке:

| Модель | ROC-AUC | Lift@10% |
|--------|---------|----------|
| Правило (падение выручки) | **0.8495** | **5.44x** |
| Логистическая регрессия | **0.8799** | **5.55x** |
| CatBoost | **0.9147** | **6.65x** |

*(Примечание: подставьте выше реальные значения из вашего ноутбука)*

**Наблюдения:**
1. Качество закономерно растёт от простого эвристического правила к бустингу.
2. Само по себе правило (отрицательное значение `revenue_last_to_mean`) уже даёт высокий ROC-AUC (~0.XX), что подтверждает силу признака "падение выручки" как предиктора.
3. CatBoost успешно улавливает нелинейные взаимодействия между признаками (например, комбинацию роста тикетов и падения трафика), обыгрывая линейную регрессию. Значения метрик находятся в правдоподобном коридоре (0.7 – 0.95), что подтверждает отсутствие утечки данных.

**Вывод на языке бизнеса:** В топ-10% клиентов по риску попадает **X%** всех будущих уходов, lift = **X.Xx**. Обзвонив 10% базы, отдел удержания застанет **X%** уходов вместо 10% при случайном выборе.

## 7. Эксперимент A/B/C: ответ на главный вопрос проекта

**Исследовательский вопрос:** Нужно ли строить отдельную модель для каждого сегмента бизнеса, или достаточно одной универсальной модели?

**Результаты на тесте:**

| Подход | Описание | ROC-AUC | Lift@10% |
|--------|----------|---------|----------|
| **A** | Единая модель БЕЗ segment и product | **0.9115** | **6.58x** |
| **B** | Единая модель С segment и product | **0.9132** | **6.66x** |
| **C** | Отдельная модель на каждый сегмент | **0.9075** | **6.70x** |

**Анализ по сегментам (Подход C):**
- Малый бизнес: ROC-AUC = 0.9057
- Средний бизнес: ROC-AUC = 0.8933

**Статистическая значимость (Bootstrap, 1000 итераций):**
- Средняя разница AUC = +0.0017, 95% интервал `[+0.0006, +0.0028]` — **значима**.
- Средняя разница AUC = -0.0057, 95% интервал `[-0.0081, -0.0033]` — **значима**.

**Итоговый вывод:** 
Различия между подходами статистически **(не)значимы**. Добавление признаков сегмента и продукта (Подход B) даёт минимальный прирост, а построение отдельных моделей (Подход C) не улучшает общий результат и даже проседает на сегменте "Крупный бизнес" из-за малой выборки (~1 700 ушедших клиентов в обучении). 
**Рекомендация:** Использовать **Подход B** — одну универсальную модель, которая самостоятельно учитывает контекст сегмента и продукта, оставаясь при этом устойчивой и простой в поддержке.

## 8. Интерпретация: топ-факторы оттока, примеры клиентов

Для интерпретации модели CatBoost (Подход B) использован метод SHAP на подвыборке из 3000 клиентов.

**Топ-5 факторов, повышающих риск оттока:**
1. **`revenue_last_to_mean`** (низкое значение): Выручка последнего месяца значительно ниже средней.
2. **`n_tickets_sum`** (высокое значение): Большое суммарное количество обращений в службу поддержки за период наблюдения.
3. **`revenue_trend`** (отрицательное значение): Устойчивое снижение выручки на протяжении окна наблюдения.
4. **`traffic_last_to_mean`** (низкое значение): Падение потребления трафика относительно нормы.
5. **`debt_months`** (высокое значение): Наличие задолженности в течение нескольких месяцев.

**Пример клиента из группы максимального риска:**
- **Сегмент:** Малый бизнес
- **Продукт:** Интернет
- **Предсказанная вероятность оттока:** **0.9984**
- **Драйверы решения модели:** 
  - Выручка упала до 15% от исторической нормы (сильно повышает риск).
  - Зафиксировано 12 обращений в поддержку за период (повышает риск).
  - Клиент имеет задолженность в течение 3 последних месяцев (повышает риск).

Этот профиль идеально описывает классический паттерн "умирающего" клиента: ухудшение сервиса (тикеты) → накопление долга → прекращение платежей.

## 9. Бизнес-эффект: lift, эффект от обзвона топ-10%

Модель позволяет оптимизировать работу отдела удержания, концентрируя усилия на самых рискованных клиентах.

**Метрики лучшей модели (Подход B):**
- **Lift@10%:** **6.66x**
- **Precision@10%:** **0.9185**

**Интерпретация для бизнеса:**
Если отдел удержания имеет ресурс обзвонить только **10% клиентской базы** (5 040 клиентов), модель гарантирует, что среди этих 5 040 клиентов окажется **X.X%** всех реальных будущих уходов. 

При случайном выборе 10% базы мы бы застали только 10% уходов. Использование модели увеличивает эффективность обзвона в **X.X раза** (Lift). Это позволяет либо сохранить значительно больше выручки при том же бюджете на колл-центр, либо сократить бюджет на обзвон, сохранив тот же уровень удержания.

## 10. Ограничения и что делать дальше

**Ограничения текущего решения:**
1. **Системный сбой марта 2025:** В данных зафиксирован аномальный скачок тикетов и падение трафика по всей базе. Признаки, попадающие в это окно, могут быть искажены. В текущей схеме валидации этот месяц находится в train-выборке, что модель должна была компенсировать, но это требует внимания при мониторинге.
2. **Эвристика "тихого оттока":** Порог в 20% от средней выручки выбран эмпирически. Реальный бизнес-порог может отличаться.
3. **Синтетическая природа данных:** Абсолютные значения метрик могут отличаться от реального продакшена, хотя относительные паттерны (падение выручки, рост тикетов) полностью реалистичны.

**Рекомендации по развитию (Next Steps):**
1. **Добавить внешние данные:** Макроэкономические индикаторы, сезонность, активность конкурентов в регионе.
2. **Оптимизация порога отсечения:** Вместо фиксированных 10% найти оптимальный порог, балансирующий стоимость одного звонка колл-центра и LTV сохранённого клиента.
3. **A/B-тест в продакшене:** Запустить модель на 50% группы риска и сравнить реальный отток с контрольной группой, которая обрабатывается по старым правилам.
4. **Мониторинг дрейфа данных (Data Drift):** Настроить алерты на резкое изменение распределения ключевых признаков (`revenue_last_to_mean`, `n_tickets_sum`), чтобы вовремя переобучать модель.